<a href="https://colab.research.google.com/github/AniShriyak/Bio-Inspired-Lab/blob/main/BIS_GENETIC_ALGORITHM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math

# ---------------------------------------------------------
# 1. Problem Data
# ---------------------------------------------------------

# Coordinates of locations
# Location 0 = depot
locations = {
    0: (0, 0),    # Depot
    1: (2, 3),
    2: (5, 4),
    3: (1, 7),
    4: (6, 8),
    5: (8, 2),
    6: (3, 9),
    7: (9, 6),
    8: (4, 1),
    9: (7, 9),
    10: (10, 3)
}


# ---------------------------------------------------------
# 2. Calculate distance between two locations
# ---------------------------------------------------------

def distance(a, b):
    x1, y1 = locations[a]
    x2, y2 = locations[b]

    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)


# ---------------------------------------------------------
# 3. Calculate total route distance
# ---------------------------------------------------------

def route_distance(route):

    total_distance = 0

    # Start from depot
    previous = 0

    # Visit every customer
    for customer in route:
        total_distance += distance(previous, customer)
        previous = customer

    # Return to depot
    total_distance += distance(previous, 0)

    return total_distance


# ---------------------------------------------------------
# 4. Fitness Function
# ---------------------------------------------------------

def fitness(route):

    # We want to MINIMIZE distance.
    # GA normally maximizes fitness,
    # so use inverse distance.

    return 1 / route_distance(route)


# ---------------------------------------------------------
# 5. Initialize Population
# ---------------------------------------------------------

def create_population(population_size):

    population = []

    customers = list(locations.keys())
    customers.remove(0)  # remove depot

    for _ in range(population_size):

        route = customers.copy()

        random.shuffle(route)

        population.append(route)

    return population


# ---------------------------------------------------------
# 6. Selection
# ---------------------------------------------------------

def selection(population):

    # Tournament selection

    tournament_size = 5

    tournament = random.sample(population, tournament_size)

    winner = max(tournament, key=fitness)

    return winner


# ---------------------------------------------------------
# 7. Crossover
# ---------------------------------------------------------

def crossover(parent1, parent2):

    size = len(parent1)

    # Choose crossover points
    start, end = sorted(
        random.sample(range(size), 2)
    )

    # Create child
    child = [None] * size

    # Copy section from parent 1
    child[start:end] = parent1[start:end]

    # Fill remaining positions from parent 2
    current_position = end

    for gene in parent2:

        if gene not in child:

            if current_position >= size:
                current_position = 0

            child[current_position] = gene

            current_position += 1

    return child


# ---------------------------------------------------------
# 8. Mutation
# ---------------------------------------------------------

def mutation(route, mutation_rate=0.02):

    route = route.copy()

    for i in range(len(route)):

        if random.random() < mutation_rate:

            j = random.randint(0, len(route) - 1)

            # Swap two customers
            route[i], route[j] = route[j], route[i]

    return route


# ---------------------------------------------------------
# 9. Genetic Algorithm
# ---------------------------------------------------------

def genetic_algorithm(
        population_size=100,
        generations=200,
        mutation_rate=0.02
):

    # Initialize population
    population = create_population(population_size)

    best_route = None
    best_distance = float("inf")

    # -----------------------------------------------------
    # Main GA loop
    # -----------------------------------------------------

    for generation in range(generations):

        # Evaluate population
        population.sort(key=route_distance)

        # Store best solution
        current_best = population[0]
        current_distance = route_distance(current_best)

        if current_distance < best_distance:

            best_route = current_best.copy()
            best_distance = current_distance

        # New population
        new_population = []

        # Elitism:
        # keep best solution
        new_population.append(current_best.copy())

        # Generate offspring
        while len(new_population) < population_size:

            parent1 = selection(population)
            parent2 = selection(population)

            child = crossover(parent1, parent2)

            child = mutation(
                child,
                mutation_rate
            )

            new_population.append(child)

        population = new_population

        # Display progress
        if generation % 50 == 0:

            print(
                f"Generation {generation}: "
                f"Best Distance = {best_distance:.2f}"
            )

    return best_route, best_distance


# ---------------------------------------------------------
# 10. Run the algorithm
# ---------------------------------------------------------

best_route, best_distance = genetic_algorithm(
    population_size=100,
    generations=500,
    mutation_rate=0.02
)


# ---------------------------------------------------------
# 11. Display result
# ---------------------------------------------------------

print("\nBest Route:")

print("Depot", end=" → ")

for customer in best_route:
    print(customer, end=" → ")

print("Depot")

print(f"\nTotal Distance: {best_distance:.2f}")


Generation 0: Best Distance = 51.91
Generation 50: Best Distance = 35.87
Generation 100: Best Distance = 35.03
Generation 150: Best Distance = 35.03
Generation 200: Best Distance = 35.03
Generation 250: Best Distance = 35.03
Generation 300: Best Distance = 35.03
Generation 350: Best Distance = 35.03
Generation 400: Best Distance = 35.03
Generation 450: Best Distance = 35.03

Best Route:
Depot → 8 → 2 → 5 → 10 → 7 → 9 → 4 → 6 → 3 → 1 → Depot

Total Distance: 35.03


This project implements a Genetic Algorithm to solve a routing optimization problem. Each candidate routing solution is represented as a chromosome containing a permutation of customer locations. An initial population of random routes is generated and evaluated using total Euclidean travel distance as the objective function. Tournament selection is used to select parent chromosomes, while permutation-based crossover generates valid offspring and swap mutation introduces genetic diversity. Elitism is used to preserve the best solution between generations. The algorithm iteratively improves the population and ultimately returns the route with the minimum distance found during the optimization process. The approach is based on established applications of genetic algorithms to the Vehicle Routing Problem, particularly the work of Baker and Ayechew (2003) and Prins (2004).
